# 01 Data profiling: Online Retail II

Measures every data quality issue before any cleaning. Read-only: the raw
Parquet file is never modified. Findings feed docs/02_data_quality_findings.md.

In [2]:
#Code cell 1: setup and load
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.2f}".format)

# Works whether the notebook runs from notebooks/ or the project root
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
df = pd.read_parquet(ROOT / "data" / "raw" / "online_retail_ii.parquet")

# Raw revenue: every line, nothing removed (used only to size each issue)
df["raw_revenue"] = df["Quantity"] * df["Price"]
TOTAL_ROWS = len(df)
GROSS_POSITIVE = df.loc[df["raw_revenue"] > 0, "raw_revenue"].sum()

print(f"Rows: {TOTAL_ROWS:,}")
print(f"Gross positive revenue: £{GROSS_POSITIVE:,.0f}")


def share(mask, label):
    """Print rows and positive revenue touched by an issue."""
    rows = mask.sum()
    rev = df.loc[mask & (df["raw_revenue"] > 0), "raw_revenue"].sum()
    print(f"{label:<45} rows {rows:>9,} ({rows / TOTAL_ROWS:6.2%})   "
          f"positive revenue £{rev:>13,.0f} ({rev / GROSS_POSITIVE:6.2%})")

Rows: 1,067,371
Gross positive revenue: £20,972,968


In [3]:
#Code cell 2: P1 overview
print(df.dtypes, "\n")
missing = df.isna().sum().to_frame("missing")
missing["share"] = missing["missing"] / TOTAL_ROWS
print(missing)

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
raw_revenue           float64
dtype: object 

             missing  share
Invoice            0   0.00
StockCode          0   0.00
Description     4382   0.00
Quantity           0   0.00
InvoiceDate        0   0.00
Price              0   0.00
Customer ID   243007   0.23
Country            0   0.00
raw_revenue        0   0.00


In [4]:
#Code cell 3: P2 cancellations
is_cancel = df["Invoice"].str.startswith("C")
neg_qty = df["Quantity"] < 0

share(is_cancel, "P2a Cancellation invoices (start with C)")
share(neg_qty, "P2b Negative quantity (any invoice)")
share(neg_qty & ~is_cancel, "P2c Negative quantity, NOT a C invoice")
share(is_cancel & ~neg_qty, "P2d C invoice with non-negative quantity")

print("\nCancelled value (absolute):",
      f"£{df.loc[is_cancel, 'raw_revenue'].abs().sum():,.0f}")
print("\nSample of P2c (negative qty without C):")
print(df.loc[neg_qty & ~is_cancel, ["Invoice", "StockCode", "Description",
                                    "Quantity", "Price", "Customer ID"]].head(10))


P2a Cancellation invoices (start with C)      rows    19,494 ( 1.83%)   positive revenue £          374 ( 0.00%)
P2b Negative quantity (any invoice)           rows    22,950 ( 2.15%)   positive revenue £            0 ( 0.00%)
P2c Negative quantity, NOT a C invoice        rows     3,457 ( 0.32%)   positive revenue £            0 ( 0.00%)
P2d C invoice with non-negative quantity      rows         1 ( 0.00%)   positive revenue £          374 ( 0.00%)

Cancelled value (absolute): £1,527,415

Sample of P2c (negative qty without C):
     Invoice StockCode      Description  Quantity  Price  Customer ID
263   489464     21733     85123a mixed       -96   0.00          NaN
283   489463     71477            short      -240   0.00          NaN
284   489467    85123A      21733 mixed      -192   0.00          NaN
470   489521     21646             None       -50   0.00          NaN
3114  489655     20683             None       -44   0.00          NaN
3162  489660     35956             lost     -10

In [5]:
#Code cell 4: P3 prices
share(df["Price"] == 0, "P3a Zero price")
share(df["Price"] < 0, "P3b Negative price")

print("\nNegative-price lines:")
print(df.loc[df["Price"] < 0, ["Invoice", "StockCode", "Description",
                               "Quantity", "Price", "InvoiceDate"]])
print("\nZero-price lines by Customer ID presence:")
print(df.loc[df["Price"] == 0, "Customer ID"].isna()
        .value_counts().rename({True: "no customer", False: "has customer"}))

P3a Zero price                                rows     6,202 ( 0.58%)   positive revenue £            0 ( 0.00%)
P3b Negative price                            rows         5 ( 0.00%)   positive revenue £            0 ( 0.00%)

Negative-price lines:
        Invoice StockCode      Description  Quantity      Price         InvoiceDate
179403  A506401         B  Adjust bad debt         1 -53,594.36 2010-04-29 13:36:00
276274  A516228         B  Adjust bad debt         1 -44,031.79 2010-07-19 11:24:00
403472  A528059         B  Adjust bad debt         1 -38,925.87 2010-10-20 12:04:00
825444  A563186         B  Adjust bad debt         1 -11,062.06 2011-08-12 14:51:00
825445  A563187         B  Adjust bad debt         1 -11,062.06 2011-08-12 14:52:00

Zero-price lines by Customer ID presence:
Customer ID
no customer     6131
has customer      71
Name: count, dtype: int64


In [6]:
#Code cell 5: P4 invoice prefixes
first_char = df["Invoice"].str[0]
prefix = np.where(first_char.str.isdigit(), "digit", first_char)
print(pd.Series(prefix).value_counts())

digit    1047871
C          19494
A              6
Name: count, dtype: int64


In [7]:
#Code cell 6: P5 non-product stock codes
# Normal product codes: 5 digits, optionally followed by letters (e.g. 85123A)
is_product_code = df["StockCode"].str.match(r"^\d{5}[A-Za-z]*$", na=False)
share(~is_product_code, "P5 Stock code not in product format")

non_product = (
    df.loc[~is_product_code]
      .groupby("StockCode")
      .agg(rows=("Invoice", "size"),
           revenue=("raw_revenue", "sum"),
           example=("Description", "first"))
      .sort_values("rows", ascending=False)
)
print(f"\nDistinct non-product codes: {len(non_product)}")
print(non_product.head(30))

P5 Stock code not in product format           rows     6,094 ( 0.57%)   positive revenue £      863,989 ( 4.12%)

Distinct non-product codes: 63
              rows     revenue                              example
StockCode                                                          
POST          2122  112,341.00                              POSTAGE
DOT           1446  322,647.47                       DOTCOM POSTAGE
M             1421  -82,796.32                               Manual
C2             282   13,386.00                             CARRIAGE
D              177  -13,484.54                             Discount
S              104   -6,065.80                              SAMPLES
BANK CHARGES   102  -35,562.63                         Bank Charges
ADJUST          67    6,835.24  Adjustment by john on 26/01/2010 16
AMAZONFEE       43 -260,763.58                           AMAZON FEE
DCGS0058        31       34.82                     MISO PRETTY  GUM
gift_0001_30    29      610.09   Dotcom

In [8]:
#Code cell 7: P6 missing Customer ID
no_cust = df["Customer ID"].isna()
share(no_cust, "P6 Missing Customer ID")
print(f"\nDistinct customers: {df['Customer ID'].nunique():,}")

P6 Missing Customer ID                        rows   243,007 (22.77%)   positive revenue £    3,229,539 (15.40%)

Distinct customers: 5,942


In [9]:
#Code cell 8: P7 duplicates and sheet overlap
dup_all = df.duplicated(keep=False)   # every copy
dup_extra = df.duplicated()           # only the extra copies
share(dup_extra, "P7a Extra copies of identical rows")

# Overlap window between the two Excel sheets
window = (df["InvoiceDate"] >= "2010-12-01") & (df["InvoiceDate"] < "2010-12-10")
share(window, "P7b Rows dated 1-9 Dec 2010")
share(window & dup_extra, "P7c Extra copies inside that window")
share(~window & dup_extra, "P7d Extra copies outside that window")

print("\nExample duplicate group:")
first_dup = df.loc[dup_all].sort_values(["Invoice", "StockCode"]).head(6)
print(first_dup[["Invoice", "StockCode", "Quantity", "Price", "InvoiceDate", "Customer ID"]])

P7a Extra copies of identical rows            rows    34,335 ( 3.22%)   positive revenue £      496,334 ( 2.37%)
P7b Rows dated 1-9 Dec 2010                   rows    45,046 ( 4.22%)   positive revenue £      877,705 ( 4.18%)
P7c Extra copies inside that window           rows    22,844 ( 2.14%)   positive revenue £      440,074 ( 2.10%)
P7d Extra copies outside that window          rows    11,491 ( 1.08%)   positive revenue £       56,260 ( 0.27%)

Example duplicate group:
    Invoice StockCode  Quantity  Price         InvoiceDate  Customer ID
379  489517     21491         1   1.95 2009-12-01 11:34:00    16,329.00
391  489517     21491         1   1.95 2009-12-01 11:34:00    16,329.00
365  489517     21821         1   3.75 2009-12-01 11:34:00    16,329.00
386  489517     21821         1   3.75 2009-12-01 11:34:00    16,329.00
363  489517     21912         1   3.75 2009-12-01 11:34:00    16,329.00
371  489517     21912         1   3.75 2009-12-01 11:34:00    16,329.00


In [10]:
#Code cell 9: P8 extremes
print("Quantity percentiles:")
print(df["Quantity"].quantile([0, .01, .05, .5, .95, .99, .999, 1]))
print("\nPrice percentiles:")
print(df["Price"].quantile([0, .01, .5, .99, .999, 1]))

print("\nTop 8 lines by absolute quantity:")
print(df.reindex(df["Quantity"].abs().sort_values(ascending=False).index)
        [["Invoice", "StockCode", "Description", "Quantity", "Price", "InvoiceDate", "Customer ID"]]
        .head(8))

Quantity percentiles:
0.00   -80,995.00
0.01        -3.00
0.05         1.00
0.50         3.00
0.95        30.00
0.99       100.00
1.00       500.00
1.00    80,995.00
Name: Quantity, dtype: float64

Price percentiles:
0.00   -53,594.36
0.01         0.21
0.50         2.10
0.99        18.00
1.00       217.02
1.00    38,970.00
Name: Price, dtype: float64

Top 8 lines by absolute quantity:
         Invoice StockCode                         Description  Quantity  Price         InvoiceDate  Customer ID
1065883  C581484     23843         PAPER CRAFT , LITTLE BIRDIE    -80995   2.08 2011-12-09 09:27:00    16,446.00
1065882   581483     23843         PAPER CRAFT , LITTLE BIRDIE     80995   2.08 2011-12-09 09:15:00    16,446.00
587080    541431     23166      MEDIUM CERAMIC TOP STORAGE JAR     74215   1.04 2011-01-18 10:01:00    12,346.00
587085   C541433     23166      MEDIUM CERAMIC TOP STORAGE JAR    -74215   1.04 2011-01-18 10:17:00    12,346.00
90857     497946     37410  BLACK AND WHITE PAI

In [11]:
#Code cell 10: P9 date coverage
d = df.assign(date=df["InvoiceDate"].dt.date,
              weekday=df["InvoiceDate"].dt.day_name(),
              month=df["InvoiceDate"].dt.to_period("M"))

print("Rows by weekday:")
print(d["weekday"].value_counts()
        .reindex(["Monday", "Tuesday", "Wednesday", "Thursday",
                  "Friday", "Saturday", "Sunday"], fill_value=0))

print("\nTrading days per month and rows per month:")
monthly = d.groupby("month").agg(trading_days=("date", "nunique"),
                                 rows=("Invoice", "size"))
print(monthly)

print("\nHour range:", d["InvoiceDate"].dt.hour.min(), "to", d["InvoiceDate"].dt.hour.max())

Rows by weekday:
weekday
Monday       189084
Tuesday      196626
Wednesday    185051
Thursday     203149
Friday       153803
Saturday        402
Sunday       139256
Name: count, dtype: int64

Trading days per month and rows per month:
         trading_days   rows
month                       
2009-12            21  45228
2010-01            24  31555
2010-02            24  29388
2010-03            27  41511
2010-04            23  34057
2010-05            24  35323
2010-06            26  39983
2010-07            26  33383
2010-08            26  33306
2010-09            26  42091
2010-10            26  59098
2010-11            26  78015
2010-12            20  65004
2011-01            24  35147
2011-02            24  27707
2011-03            27  36748
2011-04            21  29916
2011-05            25  37030
2011-06            26  36874
2011-07            26  39518
2011-08            26  35284
2011-09            26  50226
2011-10            26  60742
2011-11            26  84711
2011-12    

In [12]:
#Code cell 11: P10 countries
pos = df["raw_revenue"] > 0
country = (df.loc[pos].groupby("Country")["raw_revenue"].sum()
             .sort_values(ascending=False))
print(f"Countries: {df['Country'].nunique()}")
print((country / country.sum()).head(10).map("{:.1%}".format))

Countries: 43
Country
United Kingdom    85.2%
EIRE               3.2%
Netherlands        2.6%
Germany            2.1%
France             1.7%
Australia          0.8%
Spain              0.5%
Switzerland        0.5%
Sweden             0.4%
Denmark            0.3%
Name: raw_revenue, dtype: object


In [13]:
#Code cell 12: P11 descriptions per stock code
desc_per_code = df.dropna(subset=["Description"]).groupby("StockCode")["Description"].nunique()
print(desc_per_code.describe())
print(f"\nCodes with more than one description: {(desc_per_code > 1).sum():,}")
example_code = desc_per_code.idxmax()
print(f"\nExample, {example_code}:")
print(df.loc[df["StockCode"] == example_code, "Description"].value_counts().head(8))

count   4,950.00
mean        1.33
std         0.67
min         1.00
25%         1.00
50%         1.00
75%         1.00
max         9.00
Name: Description, dtype: float64

Codes with more than one description: 1,232

Example, 20713:
Description
JUMBO BAG OWLS                  1372
missing                            1
wrongly marked. 23343 in box       1
wrongly coded-23343                1
found                              1
Found                              1
wrongly marked 23343               1
Marked as 23343                    1
Name: count, dtype: int64


   ## Follow-up checks
   Remaining non-product codes, and whether orders have exact matching cancellations.

In [14]:
#Check A: the remaining 33 non-product codes, plus the odd “C” invoice with positive quantity
print(non_product.iloc[30:])
print("\nP2d line:")
print(df.loc[is_cancel & ~neg_qty, ["Invoice", "StockCode", "Description",
                                    "Quantity", "Price", "Customer ID"]])

              rows  revenue                             example
StockCode                                                      
DCGS0070         3    25.44               CAMOUFLAGE DOG COLLAR
gift_0001_70     3    59.57  Dotcomgiftshop Gift Voucher £70.00
DCGS0068         3    17.12                   DOGS NIGHT COLLAR
SP1002           3    14.75              KID'S CHALKBOARD/EASEL
DCGS0037         2    12.72                  KEY-RING CORKSCREW
gift_0001_90     2     0.00                                None
DCGS0062         2     2.51             ROAD-RAGE CAR FRESHENER
gift_0001_60     2     0.00                                None
TEST002          2     1.00             This is a test product.
DCGS0067         1     0.00                                ebay
DCGS0039         1     0.00                                None
DCGS0036         1     0.00                                None
DCGS0044         1     2.57             HANDZ-OFF CAR FRESHENER
C3               1     0.00             

In [15]:
#Check B: how often orders have an exact matching cancellation
orders = df.loc[~is_cancel & (df["Quantity"] > 0) & df["Customer ID"].notna()]
cancels = (df.loc[is_cancel & df["Customer ID"].notna(),
                  ["Customer ID", "StockCode", "Quantity", "Invoice", "InvoiceDate"]]
             .assign(Quantity=lambda x: -x["Quantity"]))

# 1) The biggest orders: were they cancelled?
big = orders.loc[orders["Quantity"] >= 10000]
print(big.merge(cancels, on=["Customer ID", "StockCode", "Quantity"],
                how="left", suffixes=("", "_cancel"))
         [["Invoice", "StockCode", "Quantity", "Price", "Customer ID",
           "Invoice_cancel", "InvoiceDate_cancel"]])

# 2) Overall: share of cancellation lines with an exact matching order
keys = ["Customer ID", "StockCode", "Quantity"]
match = cancels.merge(orders[keys].drop_duplicates(), on=keys, how="left", indicator=True)
print("\nCancellation lines with an exact matching order:")
print(match["_merge"].value_counts(normalize=True).map("{:.1%}".format))

   Invoice StockCode  Quantity  Price  Customer ID Invoice_cancel  InvoiceDate_cancel
0   497946     37410     19152   0.10    13,902.00            NaN                 NaT
1   501534     21099     12960   0.10    13,902.00            NaN                 NaT
2   501534     21092     12480   0.10    13,902.00            NaN                 NaT
3   501534     21091     12960   0.10    13,902.00            NaN                 NaT
4   501534     21085     12744   0.10    13,902.00            NaN                 NaT
5   502269     21984     10000   0.25    17,940.00            NaN                 NaT
6   502269     21982     10000   0.25    17,940.00            NaN                 NaT
7   502269     21980     10000   0.25    17,940.00            NaN                 NaT
8   502269     21981     10000   0.25    17,940.00            NaN                 NaT
9   541431     23166     74215   1.04    12,346.00        C541433 2011-01-18 10:17:00
10  578841     84826     12540   0.00    13,256.00    

In [16]:
# Gap checks from the checklist review
zero_width = "[\u200b\ufeff]"
print("G1 Hidden characters in text columns")
for col in ["Invoice", "StockCode", "Description", "Country"]:
    s = df[col].dropna().astype(str)
    print(f"  {col:<12} non-ASCII {s.str.contains(r'[^\x00-\x7F]').sum():>6,}"
          f"   zero-width/BOM {s.str.contains(zero_width).sum():>4,}"
          f"   internal double spaces {s.str.contains(r'\S\s{2,}\S').sum():>6,}"
          f"   leading/trailing spaces {(s != s.str.strip()).sum():>6,}")

print("\nG3 Records that may be wrongly removed or missed")
neg_normal = (df["Quantity"] < 0) & ~is_cancel
print(f"  Negative qty on normal invoices: {neg_normal.sum():,}, of which price > 0: "
      f"{(neg_normal & (df['Price'] > 0)).sum():,}")
print(f"  Quantity = 0: {(df['Quantity'] == 0).sum():,}")

print("\nG5 Missing descriptions: price profile")
nd = df["Description"].isna()
print(f"  Rows: {nd.sum():,}   of which price = 0: {(nd & (df['Price'] == 0)).sum():,}")

print("\nG6 Country values")
print(" ", sorted(df["Country"].unique()))

print("\nG7 Price outliers on product lines (after cleaning)")
sales_clean = pd.read_parquet(ROOT / "data" / "processed" / "sales_clean.parquet")
p = sales_clean.loc[sales_clean["line_type"] == "product"].copy()
p["price_ratio"] = p["price"] / p.groupby("stock_code")["price"].transform("median")
print(f"  Lines priced > 5x their product's median: {(p['price_ratio'] > 5).sum():,}")
print(f"  Lines priced < 0.2x their product's median: {(p['price_ratio'] < 0.2).sum():,}")
print(p.nlargest(5, "price")[["stock_code", "product_name", "quantity", "price", "price_ratio"]])

G1 Hidden characters in text columns
  Invoice      non-ASCII      0   zero-width/BOM    0   internal double spaces      0   leading/trailing spaces      0
  StockCode    non-ASCII      0   zero-width/BOM    0   internal double spaces      0   leading/trailing spaces      1
  Description  non-ASCII     79   zero-width/BOM    0   internal double spaces 48,398   leading/trailing spaces 213,035
  Country      non-ASCII      0   zero-width/BOM    0   internal double spaces      0   leading/trailing spaces      0

G3 Records that may be wrongly removed or missed
  Negative qty on normal invoices: 3,457, of which price > 0: 0
  Quantity = 0: 0

G5 Missing descriptions: price profile
  Rows: 4,382   of which price = 0: 4,382

G6 Country values
  ['Australia', 'Austria', 'Bahrain', 'Belgium', 'Bermuda', 'Brazil', 'Canada', 'Channel Islands', 'Cyprus', 'Czech Republic', 'Denmark', 'EIRE', 'European Community', 'Finland', 'France', 'Germany', 'Greece', 'Hong Kong', 'Iceland', 'Israel', 'Italy', 

In [ ]:
"""
Step 1: Diagnose
The script stopped before saving, so data/processed/ still holds the output of your previous successful run, 
which is identical data. Run this in a notebook cell (e.g. at the end of 01_profiling.ipynb): 
"""

s = pd.read_parquet(ROOT / "data" / "processed" / "sales_clean.parquet")
key = ["invoice", "stock_code", "quantity", "price", "invoice_date", "customer_id"]

extra = s.duplicated(subset=key)                  # extra copies only
groups = s[s.duplicated(subset=key, keep=False)].sort_values(key)
print(f"Near-duplicate extra copies: {extra.sum():,}   revenue £{s.loc[extra, 'revenue'].sum():,.0f}")

# Which columns differ inside each near-duplicate group?
for col in ["description", "country", "line_type"]:
    n = groups.groupby(key, dropna=False)[col].nunique(dropna=False).gt(1).sum()
    print(f"  groups where '{col}' differs: {n:,}")

print(groups[key + ["description", "country"]].head(12).to_string())

Near-duplicate extra copies: 2   revenue £552
  groups where 'description' differs: 2
  groups where 'country' differs: 0
  groups where 'line_type' differs: 0
       invoice stock_code  quantity  price        invoice_date  customer_id                description         country
675685  554084      23298         3   4.95 2011-05-22 11:52:00        12909          BUNTING , SPOTTY   United Kingdom
675705  554084      23298         3   4.95 2011-05-22 11:52:00        12909             SPOTTY BUNTING  United Kingdom
921011  575335      23203       300   1.79 2011-11-09 13:56:00        12931  JUMBO BAG DOILEY PATTERNS  United Kingdom
921014  575335      23203       300   1.79 2011-11-09 13:56:00        12931   JUMBO BAG VINTAGE DOILY   United Kingdom
